In [2]:
import pandas as pd
import numpy as np
from pathlib import Path 
import os 
import matplotlib.pyplot as plt

In [3]:
os.getcwd()

'c:\\Users\\chodu\\Desktop\\midterm2026\\project\\source_code'

In [4]:
pathb = os.path.abspath(os.path.join(os.getcwd(), '..'))
pathb = os.path.join(pathb, "data", "raw", "billa_prices.csv")

billa = pd.read_csv(pathb)

In [5]:
pathl = os.path.abspath(os.path.join(os.getcwd(), '..'))
pathl = os.path.join(pathl, "data", "raw", "lidl_prices.csv")

lidl = pd.read_csv(pathl)

In [6]:
pathr = os.path.abspath(os.path.join(os.getcwd(), '..'))
pathr = os.path.join(pathr, "data", "raw", "rohlik_prices.csv")

rohlik = pd.read_csv(pathr)

In [7]:
pathk = os.path.abspath(os.path.join(os.getcwd(), '..'))
pathk = os.path.join(pathk, "data", "raw", "kosik_prices.csv")

kosik = pd.read_csv(pathk)

In [8]:
billa = billa.add_prefix("billa_")
lidl = lidl.add_prefix("lidl_")
kosik = kosik.add_prefix("kosik_")
rohlik = rohlik.add_prefix("rohlik_")

In [9]:
billa = billa.rename(columns={"billa_item": "item"})
lidl = lidl.rename(columns={"lidl_item": "item"})
kosik = kosik.rename(columns={"kosik_item": "item"})
rohlik = rohlik.rename(columns={"rohlik_item": "item"})

In [10]:
full = (
    billa
    .merge(lidl, on="item", how="outer")
    .merge(kosik, on="item", how="outer")
    .merge(rohlik, on="item", how="outer")
)

In [11]:
full.head()


,item,billa_title,billa_price_czk,billa_packaging,billa_price_per_unit,lidl_title,lidl_price_czk,lidl_packaging,lidl_price_per_unit,kosik_title,kosik_price_czk,kosik_packaging,kosik_price_per_unit,rohlik_title,rohlik_price_czk,rohlik_packaging,rohlik_price_per_unit
0,banany,CORNY proteinová tyčinka 30% banán 50g,51.9,50 g,1.038000,NaN,NaN,NaN,NaN,"Banán, 1 ks",6.783,170.0 g,0.039900,Banán 1 ks,6.95,cca 190 g,0.036579
1,cervena_cocka,BILLA Čočka červená 500g,29.9,500 g,0.059800,NaN,NaN,NaN,NaN,Fine Life Čočka červená,34.900,500.0 g,0.069800,Kitchin Červená čočka,33.90,500 g,0.067800
2,cesnek,"Česká Farma Česnek síťka, 200g",49.9,200 g,0.249500,Česnek,39.9,NaN,NaN,Česnek drcený porce,64.900,1.0 kg,0.064900,"Česnek ""ošklivák"" český, síť",69.90,350 g,0.199714
3,chleb,"Chléb Chalupářský, 850g kulatý",58.9,850 g,0.069294,Dřevorubecký chléb,23.9,405 g,0.059012,Český pekař Chléb konzumní,42.500,1.2 kg,0.035417,Merhautovo pekařství Chléb žitný,35.90,400 g,0.089750
4,cibule,"Česká Farma Cibule 1kg, síť",20.9,1 kg,0.020900,NaN,NaN,NaN,NaN,"Cibule žlutá, síť",19.900,1.0 kg,0.019900,"Cibule žlutá, síť",19.90,1 kg,0.019900


In [12]:
basket_items = []

for idx, row in full.iterrows():
    item_name = row['item']
    
    # Collect BOTH prices and per-unit prices from all stores
    stores = {
        'billa': {
            'price': row['billa_price_czk'],
            'per_unit': row['billa_price_per_unit']
        },
        'lidl': {
            'price': row['lidl_price_czk'],
            'per_unit': row['lidl_price_per_unit']
        },
        'kosik': {
            'price': row['kosik_price_czk'],
            'per_unit': row['kosik_price_per_unit']
        },
        'rohlik': {
            'price': row['rohlik_price_czk'],
            'per_unit': row['rohlik_price_per_unit']
        }
    }
    
    # Filter out stores with NaN prices
    valid_stores = {
        store: data for store, data in stores.items() 
        if pd.notna(data['per_unit'])
    }
    
    if not valid_stores:
        print(f"{item_name}: NO PRICES AVAILABLE")
        continue
    
    # Find cheapest by PER_UNIT price
    cheapest_store = min(valid_stores, key=lambda s: valid_stores[s]['per_unit'])
    cheapest_actual_price = valid_stores[cheapest_store]['price']
    cheapest_per_unit = valid_stores[cheapest_store]['per_unit']
    
    print(f"{item_name:20} - {cheapest_store:8}: {cheapest_actual_price:.2f} CZK (per unit: {cheapest_per_unit:.6f})")
    
    basket_items.append({
        'item': item_name,
        'unit_price': cheapest_actual_price,  # ← ACTUAL PRICE YOU PAY
        'store': cheapest_store
    })

# Convert to DataFrame
basket_df = pd.DataFrame(basket_items)


banany               - rohlik  : 6.95 CZK (per unit: 0.036579)
cervena_cocka        - billa   : 29.90 CZK (per unit: 0.059800)
cesnek               - kosik   : 64.90 CZK (per unit: 0.064900)
chleb                - kosik   : 42.50 CZK (per unit: 0.035417)
cibule               - kosik   : 19.90 CZK (per unit: 0.019900)
jogurt               - billa   : 22.90 CZK (per unit: 0.045800)
kava                 - billa   : 79.90 CZK (per unit: 0.319600)
kureci_prsa          - kosik   : 94.90 CZK (per unit: 0.189800)
maslo                - lidl    : 34.90 CZK (per unit: 0.077556)
mleko                - kosik   : 9.90 CZK (per unit: 0.009900)
mrazena_zelenina     - billa   : 19.90 CZK (per unit: 0.044222)
ovesne_vlocky        - lidl    : 11.90 CZK (per unit: 0.023800)
ovesny_napoj         - rohlik  : 27.90 CZK (per unit: 0.027900)
paprika              - kosik   : 9.49 CZK (per unit: 0.094900)
pesto                - billa   : 29.90 CZK (per unit: 0.157368)
rajcatova_omacka     - rohlik  : 19.90 CZK 

In [13]:
# Define monthly quantities
quantities = {
    'banany': 12,
    'cervena_cocka': 1,
    'cesnek': 8,
    'chleb': 4,
    'cibule': 8,
    'jogurt': 10,
    'kava': 1,
    'kureci_prsa': 4,
    'maslo': 1,
    'mleko': 4,
    'mrazena_zelenina': 3,
    'ovesne_vlocky': 1,
    'ovesny_napoj': 4,
    'paprika': 4,
    'pesto': 1,
    'rajcatova_omacka': 4,
    'repkovy_olej': 1,
    'ryze': 2,
    'sojovy_napoj': 4,
    'spagety': 4,
    'syr_eidam': 2,
    'tofu': 2,
    'tunak': 2,
    'vejce': 2
}

In [14]:
# Calculate monthly costs
basket_df['qty_per_month'] = basket_df['item'].map(quantities)
basket_df['monthly_cost'] = basket_df['unit_price'] * basket_df['qty_per_month']

print("MONTHLY GROCERY BASKET (Cheapest Option Per Item)")
print(basket_df[['item', 'unit_price', 'qty_per_month', 'monthly_cost']].to_string())

total = basket_df['monthly_cost'].sum()
print(f"TOTAL MONTHLY COST: {total:.2f} CZK")


MONTHLY GROCERY BASKET (Cheapest Option Per Item)
                item  unit_price  qty_per_month  monthly_cost
0             banany        6.95             12         83.40
1      cervena_cocka       29.90              1         29.90
2             cesnek       64.90              8        519.20
3              chleb       42.50              4        170.00
4             cibule       19.90              8        159.20
5             jogurt       22.90             10        229.00
6               kava       79.90              1         79.90
7        kureci_prsa       94.90              4        379.60
8              maslo       34.90              1         34.90
9              mleko        9.90              4         39.60
10  mrazena_zelenina       19.90              3         59.70
11     ovesne_vlocky       11.90              1         11.90
12      ovesny_napoj       27.90              4        111.60
13           paprika        9.49              4         37.96
14             pesto

In [15]:
basket_df

,item,unit_price,store,qty_per_month,monthly_cost
0,banany,6.95,rohlik,12,83.40
1,cervena_cocka,29.90,billa,1,29.90
2,cesnek,64.90,kosik,8,519.20
3,chleb,42.50,kosik,4,170.00
4,cibule,19.90,kosik,8,159.20
5,jogurt,22.90,billa,10,229.00
6,kava,79.90,billa,1,79.90
7,kureci_prsa,94.90,kosik,4,379.60
8,maslo,34.90,lidl,1,34.90
9,mleko,9.90,kosik,4,39.60


In [16]:
output_path = os.path.join('..', 'data', 'clean', 'groceries_basket.csv')
basket_df.to_csv(output_path, index=False)